In [1]:
from datetime import date
import polars as pl

In [2]:
from data_source.sheet_ids import INVOICE_STATUS, INVOICING
from data_source.make_dataset import load_gsheet_data

In [3]:
invoice_df = load_gsheet_data(INVOICING, INVOICE_STATUS)

In [19]:
invoice_df.select(
    pl.col("Month").alias("month"),
    pl.col("Report Type").alias("report_type"),
    pl.col("Sub Type").alias("sub_type"),
    pl.col("Report Name").alias("report_name"),
    pl.col("ShipOwner").alias("customer"),
    pl.col("Starting Date").alias("start_date"),
    pl.col("Ending Date").alias("end_date"),
    pl.col("Status").alias("status"),
).with_columns(invoice_name=
    pl.when(pl.col("report_type").eq("monthly"))
    .then(
        pl.col("customer").replace("MAERSKLINE", "MAERSK")
        + " - "
        + pl.col("sub_type").str.to_uppercase()
        + " - "
        + pl.col("month")
    )
    .when(pl.col("report_type").eq("oss"))
    .then(
        pl.col("report_name").str.replace(" - OSS", "")
        + " - "
        + pl.col("sub_type").str.to_uppercase()
        + " - "
        + pl.col("month")
    )
    .when(
       ( pl.col("report_type")
        .eq("si"))
         & (pl.col("report_name").eq(pl.col("customer")))
    )
    .then(
        pl.col("report_type").str.to_uppercase()
        + " - "
        + pl.col("customer")
        + " - "
        + pl.col("sub_type").str.to_uppercase()
        + " - "
        + pl.col("month")
    )
    .when(pl.col("report_type").eq("si"))
    .then(
        pl.col("report_type").str.to_uppercase()
        + " - "
        + pl.col("report_name")
        .str.split(by="-")
        .list.first()
        .str.strip_chars()
        .str.to_titlecase()
        + " "
        + pl.col("customer")
        + " - "
        + pl.col("sub_type").str.to_uppercase()
        + " - "
        + pl.col("month")
    )
    .when(pl.col("report_type").eq("sto"))
    .then(
        pl.col("sub_type").str.to_uppercase()
        + " -"
        + pl.col("report_name").str.to_titlecase()
        + " - "
        + pl.col("customer").str.to_uppercase()
        + " - "
        + pl.col("month")
    )
    .otherwise(pl.lit("Unknown Report Type"))
).collect().write_clipboard()